# ch02 平稳性与统计方法（练习版）

> 做法：按顺序做完 **任务 1 ~ 12**，每个任务下面的 `assert` 全绿就算过。
> 卡住了先看 `# 提示：`，再不行翻讲解版对应小节（每个 `@@todo` 的描述里都标了小节号）。

**本章的硬要求**：手写的 ADF 必须与 `statsmodels` 对到 **1e-9** 以内 ——
对不上就是设计矩阵的列顺序错了，不要改断言。

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# statsmodels 是本方向第一个「真库对账」对象：ADF / ACF / PACF / ARIMA / SARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf as sm_acf
from statsmodels.tsa.stattools import adfuller, pacf as sm_pacf

warnings.filterwarnings("ignore")      # ARIMA 拟合会刷 EstimationWarning，教学时不需要看

DATA = Path("data")                    # notebook 的 cwd = coding/05_timeseries
CSV = DATA / "load_curve.csv"

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

raw = pd.read_csv(CSV, encoding="utf-8-sig", parse_dates=["时间戳"])
df = raw.drop_duplicates().sort_values("时间戳").reset_index(drop=True)
print("statsmodels 已就绪 | 原始", len(raw), "行 → 去重排序后", len(df), "行")

In [ ]:
# 脚手架：下面的常量与工具函数两版都有，你只填 @@todo 块里的内容
NLAG = 48            # 本章看 ACF/PACF 的最大滞后
P_SIM = 1            # ADF 回归里的差分滞后阶数（后面会专门讨论它为什么要调）


def clean_series(x):
    """按**小时分组**的中位数做 3 倍阈值判异常，再线性插值（沿用小节的写法）。"""
    med = x.groupby(x.index.hour).transform("median")
    return x.where(x.between(0, 3 * med)).interpolate()


def lags_2d(p, d, n_obs):
    """把 Δy 的 p 个滞后拼成 (n_obs, p) 的列块。

    ⚠️ `p = 0` 是**合法档位**（`maxlag=0`，敏感性扫描 LAG_GRID 的第一档），
    但那时列表是空的，直接 `np.column_stack([])` 会抛
    `ValueError: need at least one array to concatenate` —— 所以这里退化成空块。
    """
    if p == 0:
        return np.empty((n_obs, 0))
    return np.column_stack([d[p - i:len(d) - i] for i in range(1, p + 1)])


def bartlett(n):
    """白噪声 ACF 的 Bartlett 置信带半宽：1.96/√n。

    「ACF 落在带内」= 在 5% 水平上无法拒绝「该滞后自相关为 0」。"""
    return 1.96 / np.sqrt(n)


def sim_ar1(phi, n, seed):
    """模拟 AR(1)：x_t = phi·x_{t-1} + ε_t。"""
    rng = np.random.RandomState(seed)
    eps = rng.randn(n)
    x = np.zeros(n)
    for i in range(1, n):
        x[i] = phi * x[i - 1] + eps[i]
    return x


def sim_ma1(theta, n, seed):
    """模拟 MA(1)：x_t = ε_t + theta·ε_{t-1}。"""
    rng = np.random.RandomState(seed)
    eps = rng.randn(n)
    return eps[1:] + theta * eps[:-1]


def ar_ols(x, k):
    """对 x 拟合 AR(k)（含常数项）的条件最小二乘，返回 (系数数组, 残差)。

    系数数组是 [常数, phi_1, ..., phi_k]，所以 **PACF(k) = 系数数组的最后一个**。
    """
    v = np.asarray(x, dtype=float)
    n = len(v)
    Xd = np.column_stack([np.ones(n - k)] + [v[k - i - 1:n - i - 1] for i in range(k)])
    Yd = v[k:]
    b, *_ = np.linalg.lstsq(Xd, Yd, rcond=None)
    return b, Yd - Xd @ b


LAG_GRID = (0, 1, 2, 6, 12, 24, 48)      # ADF 滞后阶数的敏感性扫描档位


def adf_row(series, p):
    """把「maxlag=p 时的手写 τ / 真库 τ / p 值」打成一个 dict —— 敏感性表的一行。

    ⚠️ 这个辅助函数只是把表格拼装写成一行；
    **τ 本身仍然由你手写的 `adf_manual` 算**（它在任务 2 里定义）。
    """
    tau_m, _, _, _, _ = adf_manual(series, p, "c")
    tau_s, pv, _, _, _, _ = sm_adf(series, maxlag=p, autolag=None)
    return dict(maxlag=p, 手写τ=tau_m, statsmodelsτ=tau_s, p值=pv)


def arma_fit(series, p, q):
    """拟合 ARMA(p,q)（序列已经差过 d 阶，所以 order 的中间位固定填 0）。"""
    return ARIMA(series, order=(p, 0, q)).fit()


def arma_row(fits, p, q):
    """把 (p,q) 这一格的 AIC / BIC 打成一个 dict —— 定阶表的一行。

    只拟合一次、从同一个结果对象上取 AIC 与 BIC，比分别 fit 两次省一半时间。
    """
    m = fits[(p, q)]
    return dict(阶=f"({p},{q})", AIC=float(m.aic), BIC=float(m.bic))


def sm_adf(x, maxlag=None, regression="c", autolag="AIC"):
    """调 statsmodels 的 ADF，把返回值统一成 (tau, p, usedlag, nobs, crit, icbest)。

    ⚠️ 两个「返回值本身就会咬你」的坑：

    ① 必须显式传 `result_object=False`。statsmodels 0.15 起 `adfuller` 的返回类型
       会从普通 tuple 悄悄切到 `ADFullerResult`（0.16 或 2027-07 之后默认切换），
       到时候 `r[1]` 这种下标取值会静默取错。见 `docs/环境依赖.md` §三·补。

    ② **返回元素的个数取决于 `autolag`**（实测）：
       `autolag=None` → 5 个元素（没有 IC 那一项）；`autolag="AIC"` → 6 个元素。
       所以 `r[5]` 在 `autolag=None` 时会抛 `IndexError: tuple index out of range`。
       这里统一补齐成 6 元组，缺失的 IC 填 nan。
    """
    r = adfuller(np.asarray(x, dtype=float), maxlag=maxlag, regression=regression,
                 autolag=autolag, result_object=False)
    icbest = float(r[5]) if len(r) > 5 else float("nan")
    return float(r[0]), float(r[1]), int(r[2]), int(r[3]), dict(r[4]), icbest


def stem_acf(ax, vals, band, title):
    """画 ACF/PACF 的「火柴梗图」+ Bartlett 置信带（上下两条红色虚线）。"""
    lags = np.arange(len(vals))
    ax.vlines(lags, 0, vals, linewidth=0.8)
    ax.scatter(lags, vals, s=9, zorder=3)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.axhline(band, color="crimson", linestyle="--", linewidth=0.8)
    ax.axhline(-band, color="crimson", linestyle="--", linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("滞后")
    ax.set_ylabel("自相关")
    return ax


print("脚手架就绪：NLAG =", NLAG, "| 随机模拟固定 seed，结果可复现")

In [ ]:
# 本章只用 A 台区（居民型）：先把 8760 点的小时序列清洗干净，再派生一条 365 点的日序列
s_a = df[df["台区编号"] == "STATION_A_01"].set_index("时间戳")["负荷值"].asfreq("h")
a = clean_series(s_a)
Y = a.to_numpy(dtype=float)                 # 小时序列（8760,）
DY = np.diff(Y)                             # 一阶差分（8759,）
D24 = Y[24:] - Y[:-24]                      # 季节差分（8736,）
DD = np.diff(DY)                            # 二阶差分 = 过差分（8758,）

DAILY = a.resample("D").mean()              # 日序列（365,）：教科书路径的载体
DV = DAILY.to_numpy(dtype=float)
DDV = np.diff(DV)                           # 日差分（364,）

print("小时序列", Y.shape, "| 均值 %.4f" % Y.mean(), "| std %.4f" % Y.std())
print("日序列", DV.shape, "| 均值 %.4f" % DV.mean(), "| std %.4f" % DV.std())
print("差分后 std：一阶 %.4f | 季节(24) %.4f | 二阶(过) %.4f" % (DY.std(), D24.std(), DD.std()))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# macOS 上 matplotlib 默认字体不含中文，不设置的话图上全是方块
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "STHeiti"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib", matplotlib.__version__, "| 中文字体 PingFang SC")

## 任务 1：平稳性的肉眼判据（§2.1）

In [ ]:
# 三段「人工序列」，用来把 ADF 的三种判决摆在一起：
#   ① 白噪声        —— 平稳的典范
#   ② 随机游走      —— 单位根，非平稳
#   ③ 趋势 + 日周期 —— 确定性成分很强，但没有单位根
RS = np.random.RandomState(42)
N_SIM = 8760
HOURS = np.arange(N_SIM)
white = RS.randn(N_SIM) * 10 + 500
rwalk = 500 + np.cumsum(RS.randn(N_SIM) * 3)
seas = 400 + 0.01 * HOURS + 40 * np.sin(2 * np.pi * HOURS / 24)

half = N_SIM // 2
# TODO(1)：（§2.1）把三段序列各自的「前半均值 / 后半均值 / 前半 std / 后半 std」收成一张表
# 提示：用一个 dict 推导：对 (名字, 序列) 的每个组合，取 x[:half] 与 x[half:] 的 mean / std
# 提示：最后套 pd.DataFrame(...).round(4)；列名要与下面的 assert 完全一致
cmp_tbl = ____
print(cmp_tbl.to_string(index=False))

assert int(cmp_tbl.shape[0]) == 3
assert abs(float(cmp_tbl.loc[0, "前半均值"]) - 500.0865) < 1e-3
assert abs(float(cmp_tbl.loc[0, "后半均值"]) - 499.9025) < 1e-3
# 随机游走的前后半均值差了 172.88 kW —— 这就是「均值随时间变」的教科书长相
assert 172.8 < abs(float(cmp_tbl.loc[1, "前半均值"]) - float(cmp_tbl.loc[1, "后半均值"])) < 172.9
# 确定性序列的 std 前后半段**完全相等**（周期振幅固定）
assert abs(float(cmp_tbl.loc[2, "前半std"]) - float(cmp_tbl.loc[2, "后半std"])) < 1e-9
print("T1 通过 -> 三段序列的「前后半段」统计已就位")

## 任务 2：手写 ADF 的 τ 统计量（§2.2）

In [ ]:
# ---- 手写 ADF 的 τ 统计量 ----
# 回归式： Δy_t = α + δ·t + β·y_{t-1} + Σ_{i=1..p} γ_i·Δy_{t-i} + ε_t
# 检验的是 H0: β = 0（有单位根）。τ = β̂ / se(β̂)，注意它**不是** t 分布，
# 必须查 Dickey-Fuller 自己的临界值表 —— 这就是为什么必须用真库对账。
def adf_manual(x, p=P_SIM, regression="c"):
    """手写 ADF 的 τ 统计量。regression: "c" 带常数项 / "ct" 再带线性趋势 / "n" 都不带。"""
    x = np.asarray(x, dtype=float)
    d = np.diff(x)
    n_obs = len(d) - p

    # TODO(2)：（§2.2）四块自变量先都备好，每块都是 (n_obs, k) 的二维列块
    # 提示：列顺序固定为 [const][trend][level][lags...]：level = x[p:]（即 y_{t-1}）
    # 提示：lags 直接调脚手架 lags_2d(p, d, n_obs)：它就是
    #       np.column_stack([d[p-i : len(d)-i] for i in range(1, p+1)])
    # 提示：⚠️ 它内部兜住了 p=0（空列表会让 column_stack 抛 "need at least one array"）
    # 提示：trend 是 np.arange(p, len(d)) 再 reshape(-1, 1)；const 是 np.ones((n_obs, 1))
    blocks = ____
    # 本回归要用哪几块：列顺序固定为 [const][trend][level][lags...]
    used = ____
    X = ____
    level_col = ____
    Yv = ____

    # TODO(3)：（§2.2）最小二乘求解 + 经典 OLS 标准误，最后得到 τ = β̂ / se(β̂)
    # 提示：解方程用 np.linalg.lstsq(X, Yv, rcond=None)，解出来的第一个元素就是 β 向量
    # 提示：σ̂² = RSS/(n−k)；se = sqrt(σ̂² · diag((XᵀX)⁻¹))；β̂ 取 level_col 那一列
    # 提示：注意：解方程用 lstsq（走 QR/SVD），但 se 那一步按定义必须用 inv(XᵀX)
    sol = ____
    beta = ____
    resid = ____
    dof = ____
    sigma2 = ____
    se = ____
    tau = ____
    return tau, dof, sigma2, float(beta[level_col]), float(se[level_col])


adf_rows = []
for nm, x in (("白噪声", white), ("随机游走", rwalk), ("趋势+日周期", seas)):
    # TODO(4)：（§2.2）对每段序列：手写一遍、真库一遍，再比较两者的绝对差
    # 提示：手写 adf_manual(x, 1, "c")；真库 sm_adf(x, maxlag=1, autolag=None)（后面就能调）
    tau, dof, s2, b, se = ____
    tau_sm, p_sm, lags_used, nobs, crit, ic = ____
    ____
adf_tbl = pd.DataFrame(adf_rows)
print(adf_tbl.round(6).to_string(index=False))

assert float(adf_tbl["绝对差"].max()) < 1e-9, "手写 τ 必须与 statsmodels 对到 1e-9 以内"
assert abs(float(adf_tbl.loc[0, "手写τ"]) - (-67.2888)) < 1e-3, "白噪声：τ 极负 → 强烈拒绝单位根"
assert abs(float(adf_tbl.loc[1, "手写τ"]) - (-0.2051)) < 1e-3, "随机游走：τ 接近 0"
assert float(adf_tbl.loc[1, "p值"]) > 0.9, "随机游走的 p 值应接近 1（无法拒绝）"
assert float(adf_tbl.loc[2, "p值"]) < 1e-6, "确定性趋势 + 日周期：ADF 也会拒绝单位根"
print("T2 通过 -> 手写 τ 最大绝对差 %.3e" % float(adf_tbl["绝对差"].max()))

## 任务 3：ADF 的两个反直觉（§2.2.1）

In [ ]:
# ---- ADF 的两个反直觉 ----
# 反直觉 ①：A 台区是**强确定性日周期 + 噪声**，ADF 却把它判成「平稳」。
# 反直觉 ②：同一个序列，换个 maxlag，结论直接翻脸。
# TODO(5)：（§2.2.1）同一个 A 台区序列，分别在 maxlag=1 与 maxlag=24 下算 τ
# 提示：手写 adf_manual(Y, p, "c")[0] 就是 τ；真库用 sm_adf(Y, maxlag=p, autolag=None)
tau_a1, dof_a1, _, _, _ = ____
tau_a1_sm, p_a1, _, _, _, _ = ____
tau_a24, _, _, _, _ = ____
print(f"A 台区原序列  maxlag=1  → 手写τ {tau_a1:.4f} | statsmodels {tau_a1_sm:.4f} | p {p_a1:.6f}")
print(f"A 台区原序列  maxlag=24 → 手写τ {tau_a24:.4f}")

# TODO(6)：（§2.2.1）把 7 个 maxlag 档位的手写 τ、真库 τ、p 值列成一张敏感性表
# 提示：用列表推导：dict(maxlag=p, 手写τ=..., statsmodelsτ=..., p值=...) for p in (0,1,2,6,12,24,48)
# 提示：再补一列布尔量「拒绝单位根」= p 值 < 0.05
lag_sens = ____
lag_sens["拒绝单位根"] = ____
print(lag_sens.round(6).to_string(index=False))

# TODO(7)：（§2.2.1）让真库自己用 AIC 选滞后阶（autolag="AIC"），看看它选了几阶
tau_auto, p_auto, lag_auto, nobs_auto, crit_auto, ic_auto = ____
print(f"\nstatsmodels 默认（autolag='AIC'）→ 选了滞后 {lag_auto} | τ {tau_auto:.4f}"
      f" | p {p_auto:.6f} | 10% 临界值 {crit_auto['10%']:.4f}")

assert abs(tau_a1 - (-37.2580)) < 1e-3 and abs(tau_a24 - (-1.9345)) < 1e-3
assert abs(float(adf_manual(Y, 24, "c")[0] - sm_adf(Y, maxlag=24, autolag=None)[0])) < 1e-9
assert bool(lag_sens.loc[lag_sens["maxlag"] == 24, "拒绝单位根"].iloc[0]) is False
assert bool(lag_sens.loc[lag_sens["maxlag"] == 1, "拒绝单位根"].iloc[0]) is True
assert int(lag_sens["拒绝单位根"].sum()) == 6, "7 个 maxlag 里有 6 个拒绝、唯独 24 不拒绝"
assert lag_auto == 32 and float(p_auto) < 1e-4
print("T3 通过 -> ADF 的判决随 maxlag 翻脸：24 → 不拒绝（p=0.316），其余 6 档全部拒绝")

## 任务 4：差分三兄弟与过差分判据（§2.3）

In [ ]:
# ---- 差分三兄弟：一阶 / 季节(24) / 过差分 ----
# 「过差分」的判据有两条，都很硬：
#   ① 方差不再下降反而变大；② ACF1 从正转负（差分引入的负相关叠加过头了）
# TODO(8)：（§2.3）把「原序列 / 一阶差分 / 季节差分(24) / 二阶差分(过)」四段序列的
# 提示：std 用 v.std()；ACF1 用 float(sm_acf(v, nlags=1, fft=True)[1])，ACF24 同理
# 提示：四条记录拼成 list 再套 pd.DataFrame(...).round(6)
diff_tbl = ____
____
# TODO(9)：（§2.3）算两条「方差比」：一阶/原 与 二阶/一阶（后者 > 1 就是过差分的铁证）
ratio_first = ____
ratio_second = ____
print(f"\n一阶/原 = {ratio_first:.6f}  |  二阶/一阶 = {ratio_second:.6f}"
      f"  ← 大于 1 说明差分过头了")

assert abs(float(diff_tbl.loc[1, "std"]) - 36.9689) < 1e-3
assert abs(float(diff_tbl.loc[3, "std"]) - 43.0534) < 1e-3
assert float(diff_tbl.loc[3, "std"]) > float(diff_tbl.loc[1, "std"]), "过差分后方差变大"
assert float(diff_tbl.loc[1, "ACF1"]) > 0.3 and float(diff_tbl.loc[3, "ACF1"]) < -0.3
assert abs(float(DD.std() / DY.std()) - 1.164585) < 1e-5
assert abs(float(DY.std() / Y.std()) - 0.512112) < 1e-5
print("T4 通过 -> 过差分：std 36.9689 → 43.0534（比值 1.1646），ACF1 0.3219 → −0.3149 转负")

## 任务 5：手写 ACF 与 Bartlett 带（§2.4）

In [ ]:
# ---- 手写 ACF ----
# 口径要点：分子分母都用**离差**，且分母是「全部离差平方和」而不是「去掉 k 个之后的和」。
# 后者（分母用 n−k）叫无偏 ACF，statsmodels 的 adjusted 参数控制的就是它。
def acf_manual(x, nlags):
    """手写 ACF，与 statsmodels 的默认（adjusted=False）口径一致。"""
    v = np.asarray(x, dtype=float) - np.mean(x)
    # TODO(10)：（§2.4）算分母：**全部 n 项**的离差平方和（不是只留 n−k 项的无偏口径）
    denom = ____
    # TODO(11)：（§2.4）逐阶算 ACF，lag 0 固定为 1.0
    # 提示：分子是 v[k:] 与 v[:-k] 的点积，除以 denom；用列表推导一次算出 lag 1..nlags
    ac = ____
    return ac


ac_dy = acf_manual(DY, NLAG)
ac_dy_sm = np.asarray(sm_acf(DY, nlags=NLAG, fft=True))
band_dy = bartlett(len(DY))
n_over_dy = int(np.sum(np.abs(ac_dy[1:]) > band_dy))
print(f"ACF 最大绝对差 {np.max(np.abs(ac_dy - ac_dy_sm)):.3e} | Bartlett 带 ±{band_dy:.6f}")
print(f"一阶差分后：ACF1 {ac_dy[1]:+.6f} | ACF24 {ac_dy[24]:+.6f} | ACF48 {ac_dy[48]:+.6f}")
print(f"超带滞后数 {n_over_dy}/{NLAG}  ← 48 个里有 46 个还超带，日周期根本没处理掉")

assert float(np.max(np.abs(ac_dy - ac_dy_sm))) < 1e-12
assert abs(ac_dy[1] - 0.321853) < 1e-6 and abs(ac_dy[24] - 0.668394) < 1e-6
assert abs(ac_dy[48] - 0.657224) < 1e-6
assert n_over_dy == 46
assert abs(band_dy - 0.020943) < 1e-6
print("T5 通过 -> ACF 与 statsmodels 一致到 1e-16 量级；一阶差分后仍 46/48 超带")

## 任务 6：手写 PACF 的两种定义（§2.5）

In [ ]:
# ---- 手写 PACF：两种等价定义 ----
# 定义 A（递推）：Durbin-Levinson 递推，O(nlags²)
# 定义 B（回归）：PACF(k) 就是 AR(k) 条件最小二乘里**最后一个系数**
def pacf_levinson(x, nlags):
    """Durbin-Levinson 递推。out[0] 按 statsmodels 的约定固定为 1.0。"""
    r = acf_manual(x, nlags)
    phi = np.zeros((nlags + 1, nlags + 1))
    out = np.zeros(nlags + 1)
    out[0] = 1.0
    # TODO(12)：（§2.5）k=1 的初值：反射系数 phi[1,1] 与 out[1] 都等于 r[1]
    phi[1, 1] = ____
    out[1] = ____
    for k in range(2, nlags + 1):
        # TODO(13)：（§2.5）第 k 阶反射系数 phi[k,k] = 分子 / 分母
        # 提示：分子 = r[k] − Σ_{j=1..k−1} phi[k−1,j]·r[k−j]；分母 = 1 − Σ phi[k−1,j]·r[j]
        # 提示：用 sum(... for j in range(1, k)) 一次求和，避免再写一层 for
        num = ____
        den = ____
        phi[k, k] = ____
        for j in range(1, k):
            # TODO(14)：（§2.5）Levinson 更新式：用第 k 阶反射系数把前 k−1 个系数修正一遍
            phi[k, j] = ____
        out[k] = phi[k, k]
    return out


pc_lev = pacf_levinson(DY, NLAG)
# TODO(15)：（§2.5）用「AR(k) 回归的末系数」这条等价定义再算一遍 PACF，用来互验
# 提示：一行列表推导：ar_ols(DY, k) 返回 (系数数组, 残差)，取 [0][-1] 就是第 k 个偏自相关
pc_reg = ____
pc_sm = np.asarray(sm_pacf(DY, nlags=NLAG, method="ldb"))

d_lev_sm = float(np.max(np.abs(pc_lev - pc_sm)))
d_lev_reg = float(np.max(np.abs(pc_lev - pc_reg)))
print(f"Levinson  vs statsmodels.pacf  max|diff| = {d_lev_sm:.3e}")
print(f"Levinson  vs 「AR(k) 末系数」  max|diff| = {d_lev_reg:.3e}")
print(f"PACF 前 8（含 lag0）：{np.round(pc_lev[:9], 6).tolist()}")

assert d_lev_sm < 1e-12, "递推法与 statsmodels 应该一致到 1e-12 以内"
assert 1e-3 < d_lev_reg < 5e-2, "两种「定义」在本样本上并不严格相等 —— 见 §2.5.1"
assert abs(d_lev_reg - 1.179e-02) < 1e-3
assert abs(pc_lev[1] - 0.321853) < 1e-6 and abs(pc_lev[3] - (-0.299056)) < 1e-6
print(f"T6 通过 -> 递推法对得上真库（{d_lev_sm:.1e}），但「回归定义」差了 {d_lev_reg:.3e}")

## 任务 7：AR / MA 的拖尾与截尾（§2.6）

In [ ]:
# ---- AR / MA 的 ACF 形态：一个拖尾、一个截尾 ----
# 理论：AR(1) 的 ACF_k = φ^k（**拖尾**，指数衰减不截断）
#       MA(1) 的 ACF_1 = θ/(1+θ²)，ACF_k = 0 (k ≥ 2)（**截尾**）
shape_rows = []
for phi in (0.9, 0.5, -0.9):
    # TODO(16)：（§2.6）模拟 AR(1) 并取它的 ACF1..4（n=20000 让抽样误差足够小）
    xs = ____
    acs = ____
    shape_rows.append(dict(模型=f"AR(1) φ={phi:+.1f}",
                           **{f"ACF{k}": acs[k] for k in range(1, 5)},
                           理论=f"[{', '.join(f'{phi ** k:.4f}' for k in range(1, 5))}]"))
for th in (0.9, 0.5, -0.9):
    # TODO(17)：（§2.6）模拟 MA(1) 并取它的 ACF1..4 —— 重点看 lag 2 之后是否归零
    xs = ____
    acs = ____
    shape_rows.append(dict(模型=f"MA(1) θ={th:+.1f}",
                           **{f"ACF{k}": acs[k] for k in range(1, 5)},
                           理论=f"[{th / (1 + th ** 2):.4f}, 0, 0, 0]"))
shape_tbl = pd.DataFrame(shape_rows)
print(shape_tbl.round(4).to_string(index=False))

assert int(shape_tbl.shape[0]) == 6
assert abs(float(shape_tbl.loc[0, "ACF1"]) - 0.9018) < 1e-3
assert abs(float(shape_tbl.loc[0, "ACF4"]) - 0.6573) < 1e-3, "AR(1) φ=0.9 的 ACF4 应约 0.6561（拖尾）"
assert abs(float(shape_tbl.loc[3, "ACF1"]) - 0.504) < 1e-3
assert abs(float(shape_tbl.loc[3, "ACF2"])) < 0.01, "MA(1) 在 lag≥2 应截尾到 0"
assert abs(float(shape_tbl.loc[5, "ACF1"]) - (-0.497)) < 1e-3, "θ<0 时 MA(1) 的 ACF1 为负"
print("T7 通过 -> AR 拖尾、MA 截尾；用 lag 2 之后是否归零一眼分辨")

## 任务 8：日序列定阶 AIC / BIC（§2.7）

In [ ]:
# ---- ARIMA 定阶：AIC / BIC 网格 ----
# 载体换成 365 点的**日序列**：小时序列的日周期太强，不成季节差分根本定不出低阶 ARMA。
tau_d_c, p_d_c, lag_d_c, nobs_d_c, crit_d_c, ic_d_c = sm_adf(DV, autolag="AIC")
tau_d_ct, p_d_ct, lag_d_ct, nobs_d_ct, crit_d_ct, ic_d_ct = sm_adf(DV, autolag="AIC", regression="ct")
tau_dd, p_dd, lag_dd, _, _, _ = sm_adf(DDV, autolag="AIC")
print(f"日序列 ADF（含常数）     滞后 {lag_d_c:2d} | τ {tau_d_c:8.4f} | p {p_d_c:.6f}")
print(f"日序列 ADF（常数 + 趋势）滞后 {lag_d_ct:2d} | τ {tau_d_ct:8.4f} | p {p_d_ct:.6f}")
print(f"日差分 ADF（含常数）     滞后 {lag_dd:2d} | τ {tau_dd:8.4f} | p {p_dd:.6f}")

# TODO(18)：（§2.7）对 p, q ∈ {0,1,2} 的 9 个组合各拟合一次 ARMA —— 键用元组 (p, q)
# 提示：脚手架里有 arma_fit(series, p, q)：一个组合只 fit 一次，省一半时间
fits = ____
# TODO(19)：（§2.7）把 9 个组合的 AIC / BIC 列成表，并标出各自的「最优」行
# 提示：用 arma_row(fits, p, q) 生成 9 行 dict，再套 pd.DataFrame(...)
# 提示：最后加两个布尔列：AIC 列是否等于最小值、BIC 列是否等于最小值
order_tbl = ____
order_tbl["AIC最优"] = ____
order_tbl["BIC最优"] = ____
print("\n" + order_tbl.round(4).to_string(index=False))

best_aic = order_tbl.loc[order_tbl["AIC"].idxmin(), "阶"]
best_bic = order_tbl.loc[order_tbl["BIC"].idxmin(), "阶"]
params_12 = np.asarray(ARIMA(DDV, order=(1, 0, 2)).fit().params)
# ⚠️ 用「阶」这一列按名字取值，而不是 .loc[2] —— 行序是 (0,0),(0,1),(0,2),(1,0)…，
# 位置下标 2 拿到的是 (0,2)，不是 (1,2)。这类错位在宽表里极常见。
aic_12 = float(order_tbl.loc[order_tbl["阶"] == "(1,2)", "AIC"].iloc[0])
bic_12 = float(order_tbl.loc[order_tbl["阶"] == "(1,2)", "BIC"].iloc[0])
print(f"\nAIC 最优 {best_aic} | BIC 最优 {best_bic}"
      f" | ARMA(1,2) AIC {aic_12:.4f} 参数 {np.round(params_12, 6).tolist()}")

assert abs(tau_d_c - (-1.8122)) < 1e-3 and float(p_d_c) > 0.3
assert abs(tau_dd - (-5.7997)) < 1e-3 and float(p_dd) < 1e-6
assert best_aic == "(1,2)" and best_bic == "(1,2)", "AIC 与 BIC 在本数据上给出同一个阶"
assert abs(aic_12 - 3214.0624) < 1e-3
assert abs(bic_12 - 3233.5482) < 1e-3
print("T8 通过 -> 日序列 ADF p=0.374 不拒绝、差分后 p≈0；AIC/BIC 一致选出 ARMA(1,2)")

## 任务 9：残差 Ljung-Box 检验（§2.7.1）

In [ ]:
# ---- 残差检验：定完阶还不算完，残差必须是白噪声 ----
# 这是本章最重要的「负结果」：ARMA(1,2) 的 AIC 已经是最小，但残差并不白。
# TODO(20)：（§2.7.1）取出 ARMA(1,2) 的残差，并算它的 ACF 与 Bartlett 带
# 提示：best = ARIMA(DDV, order=(1,0,2)).fit()；取 np.asarray(best.resid)[3:]（掐掉前 3 个 burn-in）
best = ____
resid = ____
ac_res = ____
band_res = ____
# TODO(21)：（§2.7.1）对残差做 Ljung-Box 白噪声检验（滞后 5 / 10 / 20）
# 提示：acorr_ljungbox(resid, lags=[5, 10, 20], return_df=True)，结果里有 lb_pvalue 列
lb = ____
print(f"残差 ACF1..5 {np.round(ac_res[1:6], 6).tolist()} | 带 ±{band_res:.6f}")
print(f"残差均值 {resid.mean():.6f} | std {resid.std():.6f}")
print("Ljung-Box 检验：")
print(lb.round(6).to_string())
print(f"\n日差分序列的 ACF7 = {float(sm_acf(DDV, nlags=7, fft=True)[7]):+.6f}"
      f" | ACF14 = {float(sm_acf(DDV, nlags=14, fft=True)[14]):+.6f}"
      f"  ← 周周期还在残差里")

assert int(np.sum(np.abs(ac_res[1:]) > band_res)) == 5, "10 个滞后里有 5 个超带"
p5 = float(lb.loc[5, "lb_pvalue"])
assert p5 < 0.01, "lag=5 的 Ljung-Box 已拒绝白噪声"
assert abs(p5 - 1.425489e-03) < 1e-6
assert abs(float(lb.loc[10, "lb_pvalue"]) - 4.671439e-15) < 1e-16
print(f"T9 通过 -> Ljung-Box p(5)={p5:.3e}：残差**不是**白噪声，说明还有周期没建模")

## 任务 10：加季节项 SARIMA（§2.8）

In [ ]:
# ---- SARIMA：把「季节」当成一个额外的 ARMA 层 ----
# SARIMAX(order=(p,d,q), seasonal_order=(P,D,Q,s))：季节块作用在「滞后 s 的整数倍」上
# 这里 s=7：日序列的周周期（实测 ACF7 = 0.428，正好被非季节 ARMA 漏掉）
# TODO(22)：（§2.8）把「周周期」当成一层额外的 ARMA：拟合 SARIMAX(1,0,2)(1,0,0,7)
# 提示：seasonal_order 是 (P, D, Q, s)，s=7 就是周周期；拟合时传 disp=False 关掉迭代日志
# 提示：序列已经差过一阶，所以 D 取 0；再传 enforce_stationarity=enforce_invertibility=False
sar = ____
params_sar = np.asarray(sar.params)
print(f"SARIMAX(1,0,2)(1,0,0,7)  AIC {sar.aic:.4f} | BIC {sar.bic:.4f}")
print(f"参数 [ar.L1, ma.L1, ma.L2, ar.S.L7, sigma2] = {np.round(params_sar, 6).tolist()}")
print(f"对比 非季节 ARMA(1,2)    AIC {aic_12:.4f}"
      f"  →  季节项省下 {aic_12 - float(sar.aic):.4f}")

assert abs(float(sar.aic) - 3050.7332) < 1e-3
assert abs(float(sar.bic) - 3070.1079) < 1e-3
assert float(sar.aic) < aic_12 - 100, "加季节项应显著降低 AIC"
assert len(params_sar) == 5
assert abs(float(params_sar[3]) - 0.453191) < 1e-5, "ar.S.L7 就是「上上周同一天」的系数"
print("T10 通过 -> SARIMA 把 AIC 从 3214.06 降到 3050.73（省 163.33）")

## 任务 11~12：把结论画出来（§2.9）

In [ ]:
# ---- 图 1：三段对照序列 + 它们的 ADF 判决 ----
fig, axes = plt.subplots(3, 1, figsize=(11, 7.2), sharex=True)
for ax, (nm, x) in zip(axes, (("白噪声", white), ("随机游走", rwalk), ("趋势 + 日周期", seas))):
    ax.plot(np.arange(len(x)) / 24, x, linewidth=0.6)
    # TODO(23)：（§2.9）算这段序列的 ADF τ 与 p 值，写进标题里当"判决书"
    tau_v, p_v, _, _, _, _ = ____
    ax.set_title(f"{nm}   τ = {tau_v:.2f}   p = {p_v:.3g}", fontsize=10)
    ax.set_ylabel("取值")
axes[-1].set_xlabel("天")
fig.suptitle("三种序列与 ADF 的判决（同一把尺子，三种结果）")
fig.tight_layout()

assert True
print("图 1 已绘制：白噪声 τ=-67.29 / 随机游走 τ=-0.21 / 趋势+日周期 τ=-103.61")

In [ ]:
# ---- 图 2：ACF / PACF 面板（原序列 vs 一阶差分）----
band_y = bartlett(len(Y))
fig, axes = plt.subplots(2, 2, figsize=(12, 6.4))
stem_acf(axes[0, 0], acf_manual(Y, NLAG), band_y, "A 台区原序列：ACF（30/30 超带）")
stem_acf(axes[0, 1], pacf_levinson(Y, NLAG), band_y, "A 台区原序列：PACF（lag1 后骤降）")
stem_acf(axes[1, 0], ac_dy, band_dy, "一阶差分：ACF（lag24 仍有 0.668）")
stem_acf(axes[1, 1], pc_lev, band_dy, "一阶差分：PACF")
fig.suptitle("原序列 vs 一阶差分：ACF/PACF 面板（红虚线 = Bartlett 带 ±1.96/√n）")
fig.tight_layout()

fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4.2))
names = ["原序列", "一阶差分", "季节差分", "二阶(过)"]
# TODO(24)：（§2.9）把四段序列的 std 与 ACF1 各收成一个列表，用来画对照柱状图
# 提示：std 直接 v.std()；ACF1 用 float(sm_acf(v, nlags=1, fft=True)[1])
stds = ____
acf1s = ____
axes2[0].bar(names, stds, color=["#888", "#2a7", "#2a7", "#c33"])
axes2[0].set_title("四段序列的标准差（过差分 → 反而变大）")
axes2[0].set_ylabel("std")
axes2[1].bar(names, acf1s, color=["#888", "#2a7", "#2a7", "#c33"])
axes2[1].axhline(0, color="black", linewidth=0.6)
axes2[1].set_title("四段序列的 ACF1（过差分 → 转负）")
fig2.tight_layout()

assert abs(float(acf_manual(Y, NLAG)[24]) - 0.912160) < 1e-6
assert abs(float(pacf_levinson(Y, NLAG)[1]) - 0.868814) < 1e-6
print("图 2 已绘制：原序列 ACF lag24 = 0.9122；一阶差分后 lag24 仍 = 0.6684")

## 自查清单（能不看笔记答出来才算过）

- [ ] 12 个任务的 `assert` 全部通过
- [ ] ADF 的 τ 为什么不能查 t 分布表？三种 `regression` 的 5% 临界值各是多少？
- [ ] A 台区原序列的 ADF p 值是多少？它真的"平稳"吗？该看哪个指标才能发现不对？
- [ ] 同一个序列 `maxlag=1` 和 `maxlag=24` 的 ADF 结论分别是什么？为什么？
- [ ] 过差分的两条判据是什么？本数据的两个具体数字是多少？
- [ ] ACF 的分母用 n 还是 n−k？对应 `statsmodels` 的哪个参数？
- [ ] PACF 的 lag 0 约定是多少？两种定义为什么在小样本下不相等？
- [ ] AR / MA 的 ACF 分别有什么形态？怎么用 lag 2 之后的取值区分它们？
- [ ] 日序列的 ADF 结论和小时序列为什么相反？
- [ ] `ARMA(1,2)` 的 Ljung-Box p 值是多少？它说明什么？结构藏在哪个滞后上？
- [ ] SARIMA 的 `(P,D,Q,s)` 各管什么？本数据为什么取 `s=7`？
- [ ] 加季节项后 AIC 从多少降到多少？

## 关键真值对照（做完再对）

| 量 | 值 |
|---|---|
| 白噪声 / 随机游走 / 趋势+日周期 的 ADF τ | −67.2888 / −0.2051 / −103.6063 |
| 随机游走前后半段均值 | 396.8295 / 569.7063（差 172.88） |
| 随机游走 ACF1 / 其一阶差分 ACF1 | +0.999366 / −0.000012 |
| 手写 ADF vs statsmodels 最大绝对差 | 4.974e-14 |
| A 原序列 ADF（maxlag=1 / 24） | τ −37.2580 / −1.9345；p 0.000000 / **0.315992** |
| A 原序列 ADF（autolag=AIC） | 滞后 32、τ −4.9443、p 0.000029 |
| 一阶差分 ADF | τ −56.5009、p 0.000000、std 36.9689 |
| 季节差分(24) ADF | τ −35.8735、p 0.000000、std 29.6486 |
| 过差分 std / 比值 / ACF1 | 43.0534 / 1.164585 / **−0.314890** |
| 一阶差分 ACF1 / ACF24 / ACF48 | +0.321853 / +0.668394 / +0.657224 |
| 一阶差分超带滞后数（1..48） | **46 / 48** |
| Bartlett 带（n=8759） | ±0.020943 |
| 手写 ACF vs statsmodels | 3.331e-16 |
| 手写 PACF(Levinson) vs statsmodels | 3.053e-15 |
| 手写 PACF(Levinson) vs 「AR(k) 末系数」 | **1.179e-02** |
| A 原序列 ACF1 / ACF24 | +0.868814 / +0.912160（30/30 超带） |
| A 原序列 PACF1 / PACF2 / PACF24 | +0.868814 / −0.414632 / +0.146101 |
| AR(1) φ=0.9 的 ACF1..4 | 0.9018 / 0.8123 / 0.7303 / 0.6573 |
| MA(1) θ=0.9 的 ACF1..4 | 0.5040 / 0.0081 / 0.0039 / 0.0066 |
| 日序列 ADF（c / ct） | τ −1.8122（p 0.374400）/ τ −1.4542（p 0.844277） |
| 日差分 ADF | 滞后 15、τ −5.7997、p 0.000000 |
| 日差分 ACF7 / ACF14 | +0.428114 / +0.427096 |
| ARMA(1,2) AIC / BIC | 3214.0624 / 3233.5482 |
| 残差 Ljung-Box p（lag 5 / 10 / 20） | **1.425489e-03** / 4.671439e-15 / 5.560927e-27 |
| SARIMAX(1,0,2)(1,0,0,7) AIC / BIC | **3050.7332** / 3070.1079 |
| 季节项 `ar.S.L7` | +0.453191 |
| 日序列线性趋势 | 斜率 −0.053120 kW/天、R² 0.014691 |